# Exercise: MCP Servers

This notebook guides you through setting up and running an AI Agent that will use MCP servers.
The steps include:

1. Installing required dependencies.
2. Importing necessary libraries.
3. Setting up the MCP server.
4. Setting up the language model and the AI Agent.
5. Prompting the AI Agent.

Follow the instructions in the code cells and ensure all dependencies are installed correctly before proceeding.


In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Install required dependencies
!pip3 install -r requirements.txt

In [ ]:
# Import necessary libraries for downloading models and setting up the chat system
from huggingface_hub import hf_hub_download
from mcp.client.stdio import stdio_client
from mcp import ClientSession, StdioServerParameters

from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_community.chat_models import ChatLlamaCpp
from langchain_mcp_adapters.tools import load_mcp_tools

In [ ]:
# Download the LLM, you can search in Hugging Face
model_path = hf_hub_download(
    repo_id="bartowski/Qwen2.5-7B-Instruct-GGUF",
    filename="Qwen2.5-7B-Instruct-Q4_K_M.gguf",
    force_download=False,
)

In [ ]:
# Create the LLM
llm = ChatLlamaCpp(
    model_path=model_path,
    n_gpu_layers=25,
    stop=["<|im_end|>\n"],
    n_ctx=4096,
    max_tokens=4096,
    streaming=False,
    n_batch=256,
)

In [ ]:
# Create the params
server_params = StdioServerParameters(
    command="python",
    args=["math_server.py"]
)

In [ ]:
# Get the tools
async with stdio_client(server_params) as (read, write):
  async with ClientSession(read, write) as session:
    await session.initialize()
    tools = await load_mcp_tools(session)

print("Available tools:", tools)

In [ ]:
# Create the agetnt
agent = create_agent(llm, tools)

In [ ]:
# Invoke the LLM with a new message
agent.invoke({"messages": [HumanMessage(content="what's (3 + 5) x 12?")]})